In [12]:
import pandas as pd

In [3]:
data = pd.read_csv("../data/Coffe_sales.csv")
data.head()

,hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


In [5]:
data["cash_type"].nunique()

1

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3547 entries, 0 to 3546
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   hour_of_day  3547 non-null   int64  
 1   cash_type    3547 non-null   str    
 2   money        3547 non-null   float64
 3   coffee_name  3547 non-null   str    
 4   Time_of_Day  3547 non-null   str    
 5   Weekday      3547 non-null   str    
 6   Month_name   3547 non-null   str    
 7   Weekdaysort  3547 non-null   int64  
 8   Monthsort    3547 non-null   int64  
 9   Date         3547 non-null   str    
 10  Time         3547 non-null   str    
dtypes: float64(1), int64(3), str(7)
memory usage: 304.9 KB


In [6]:
x = data.drop(columns=["cash_type","money","Date","Time"])
y = data["money"]
print(x)
print(y)

      hour_of_day    coffee_name Time_of_Day Weekday Month_name  Weekdaysort  \
0              10          Latte     Morning     Fri        Mar            5   
1              12  Hot Chocolate   Afternoon     Fri        Mar            5   
2              12  Hot Chocolate   Afternoon     Fri        Mar            5   
3              13      Americano   Afternoon     Fri        Mar            5   
4              13          Latte   Afternoon     Fri        Mar            5   
...           ...            ...         ...     ...        ...          ...   
3542           10     Cappuccino     Morning     Sun        Mar            7   
3543           14          Cocoa   Afternoon     Sun        Mar            7   
3544           14          Cocoa   Afternoon     Sun        Mar            7   
3545           15      Americano   Afternoon     Sun        Mar            7   
3546           18          Latte       Night     Sun        Mar            7   

      Monthsort  
0             3  
1  

In [7]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(2837, 7)
(710, 7)
(2837,)
(710,)


In [8]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

num_features = x.select_dtypes(include=["int64","float64"]).columns
cat_features = x.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num",num_pipeline,num_features),
    ("cat",cat_pipeline,cat_features)
])

C:\Users\nigam\AppData\Local\Temp\ipykernel_23184\2612607518.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = x.select_dtypes(include=["object"]).columns


In [10]:
from sklearn.metrics import r2_score,root_mean_squared_error

from sklearn.linear_model import (
    LinearRegression,Ridge,Lasso,ElasticNet,
    BayesianRidge,ARDRegression,SGDRegressor,
    PassiveAggressiveRegressor,HuberRegressor,
    RANSACRegressor,TheilSenRegressor
)

from sklearn.ensemble import RandomForestRegressor

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "BayesianRidge": BayesianRidge(),
    "ARDRegression": ARDRegression(),
    "SGDRegressor": SGDRegressor(max_iter=1000),
    "PassiveAggressive": PassiveAggressiveRegressor(max_iter=1000),
    "Huber": HuberRegressor(),
    "RANSAC": RANSACRegressor(),
    "TheilSen": TheilSenRegressor(),
    "RandomForest":RandomForestRegressor(n_estimators=200,max_depth=50,random_state=42,n_jobs=-1)
}

results = []

for name,model in models.items():
    try:
        pipe = Pipeline([
            ("preprocessor",preprocessor),
            ("Model",model)
        ])
        
        pipe.fit(x_train,y_train)
        pred = pipe.predict(x_test)
        
        r2 = r2_score(y_test,pred)
        rmse = root_mean_squared_error(y_test,pred)
        
        results.append((name,r2,rmse))
    
    except Exception as e:
        print(f"{name} failed as {e}")

results_df = pd.DataFrame(results,columns=["Model","R2-Score","RMSE"])
results_df = results_df.sort_values(by="RMSE",ascending=False)
results_df            

c:\Users\nigam\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveRegressor is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDRegressor(loss='epsilon_insensitive', penalty=None, learning_rate='pa1', eta0 = 1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
c:\Users\nigam\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


,Model,R2-Score,RMSE
2,Lasso,0.001433,4.779531
3,ElasticNet,0.098587,4.541073
7,PassiveAggressive,0.945488,1.116718
8,Huber,0.965117,0.893317
10,TheilSen,0.967253,0.865527
9,RANSAC,0.975063,0.755298
6,SGDRegressor,0.976304,0.736266
1,Ridge,0.976493,0.733317
4,BayesianRidge,0.976516,0.732969
0,LinearRegression,0.976516,0.732966


## RandomForestRegressor wins with 
                    (n_estimators=200,
                    max_depth=50,
                    random_state=42,
                    n_jobs=-1)